In [2]:
import numpy as np
import matplotlib.pyplot as plt




In [6]:
#Variables 
G = 6.67430e-11   # gravitational constant
#Mass of Moon
Mm = 7.342e22  # Mass of Moon (kg)=
#Mass of Earth (kg)
Me = 5.972e24
#Mass of Sun (kg)
Ms = 1.9885e30 

#Mass of 1338A Star (kg)
MA = 1.03784970719363567 * Ms

#Mass of 1338B Star (kg)
MB = 2.97388770751337850e-01 * Ms

#Mass of 1338b (gas giant planet) (kg)
Mb = 9.06017229632760055e-05 * Ms 

masses = np.array([MA, MB, Mb])

r1 = np.array([1.95196590778876217e-02, 1.32648896371446897e-04, 2.22880532978689296e-02]) #1338A Star
v1 = np.array([-7.66586168788931707e-03, 5.45640776560390306e-05, 9.13006908476301365e-03])

r2 = np.array([-6.82335050435495666e-02, -4.65900424342245798e-04, -7.78747286204819755e-02]) #1338B Star
v2 = np.array([2.67578221143398472e-02, -1.90441050119945319e-04, -3.18697187907663951e-02])

r3 = np.array([3.68709659800795730e-01, 9.75628498103004414e-03, 3.02645766911772141e-01]) #1338b Planet
v3 = np.array([-1.61533175726252566e-02, 6.31089229491667655e-05, 2.27034214206392367E-02])

y = np.concatenate([r1, v1, r2, v2, r3, v3])
print(y)

[ 1.95196591e-02  1.32648896e-04  2.22880533e-02 -7.66586169e-03
  5.45640777e-05  9.13006908e-03 -6.82335050e-02 -4.65900424e-04
 -7.78747286e-02  2.67578221e-02 -1.90441050e-04 -3.18697188e-02
  3.68709660e-01  9.75628498e-03  3.02645767e-01 -1.61533176e-02
  6.31089229e-05  2.27034214e-02]


In [ ]:
# Define the three-body system dynamics
def three_body_derivatives(t, y, masses):
    # Unpack state vector
    # Distances between bodies
    r12 = r2 - r1
    r13 = r3 - r1
    r21 = r1 - r2
    r23 = r3 - r2
    r31 = r1 - r3
    r32 = r2 - r3
    
    # Norms
    d12 = np.linalg.norm(r12)
    d13 = np.linalg.norm(r13)
    d23 = np.linalg.norm(r23)
    
    # Accelerations
    a1 = G*masses[1]*r12/d12**3 + G*masses[2]*r13/d13**3
    a2 = G*masses[0]*r21/d12**3 + G*masses[2]*r23/d23**3
    a3 = G*masses[0]*r31/d13**3 + G*masses[1]*r32/d23**3
    
    # Return derivative of state
    dydt = np.concatenate([v1, a1, v2, a2, v3, a3])
    return dydt

In [8]:
#RK4 Function
def rk4_step(f, t, y, h, masses):
    k1 = f(t, y, masses)
    k2 = f(t + h/2, y + h*k1/2, masses)
    k3 = f(t + h/2, y + h*k2/2, masses)
    k4 = f(t + h, y + h*k3, masses)

    return y + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

#Loop for RK4 integration
def integrate_rk4(f, y0, t0, tf, h, masses):
    t_values = [t0]
    y_values = [y0.copy()]
    
    t = t0
    y = y0.copy()
    
    while t < tf:
        y = rk4_step(f, t, y, h, masses)
        t += h
        
        t_values.append(t)
        y_values.append(y.copy())
    
    return np.array(t_values), np.array(y_values)


In [9]:
#Set Initial conditions 
r1_0 = np.array([0.0, 0.0, 0.0])
v1_0 = np.array([0.0, 0.0, 0.0])

r2_0 = np.array([0.0, 0.0, 0.0])
v2_0 = np.array([0.0, 0.0, 0.0])

r3_0 = np.array([0.0, 0.0, 0.0])
v3_0 = np.array([0.0, 0.0, 0.0])

y0 = np.concatenate([r1_0, v1_0, r2_0, v2_0, r3_0, v3_0])

t, y = integrate_rk4(three_body_derivatives, y0, 0.0, 1.0e6, 100.0, masses)

In [10]:
#Trajectories 
r1_traj = y[:, 0:3]
r2_traj = y[:, 6:9]
r3_traj = y[:, 12:15]